In [ ]:
import os, sys, time, subprocess
TRACE = open('/kaggle/working/trace.log', 'w')
def T(msg):
    TRACE.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n'); TRACE.flush()
    print(msg, flush=True)
T('=== CELL_1_START ===')
import torch
T(f'torch={torch.__version__} cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    T(f'  device={torch.cuda.get_device_name(0)} cc={torch.cuda.get_device_capability(0)}')
T('=== CELL_1_END ===')

In [ ]:
T('=== CELL_2_START (install) ===')
from pathlib import Path
BZ = '/kaggle/working/boltz_pkgs'
Path(BZ).mkdir(exist_ok=True)
BIN = f'{BZ}/bin/boltz'
T(f'BIN exists: {Path(BIN).exists()}')
if not Path(BIN).exists():
    T('Installing...')
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', BZ, '-q',
        'numpy==1.26.4', 'torch==2.4.0', 'torchmetrics==1.4.0', 'lightning==2.4.0',
        'boltz', 'rdkit', 'pyyaml'], capture_output=True, text=True, timeout=2400)
    T(f'install rc={r.returncode} elapsed={time.time()-t0:.0f}s')
    T(f'install stderr tail (last 1500): {r.stderr[-1500:]}')
    if Path(f'{BZ}/bin').exists():
        for f in Path(f'{BZ}/bin').iterdir(): f.chmod(0o755)
T(f'BIN exists after install: {Path(BIN).exists()}')
T('=== CELL_2_END ===')

In [ ]:
T('=== CELL_3_START (boltz --help) ===')
env = {**os.environ, 'PYTHONPATH': BZ}
r = subprocess.run([BIN, '--help'], env=env, capture_output=True, text=True, timeout=60)
T(f'boltz --help rc={r.returncode}')
T(f'  stdout[:600]: {r.stdout[:600]}')
if r.returncode != 0:
    T(f'  stderr[-1500:]: {r.stderr[-1500:]}')
T('=== CELL_3_END ===')

In [ ]:
T('=== CELL_4_START (single predict) ===')
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
yaml_str = f'version: 1\nsequences:\n- protein:\n    id: A\n    sequence: {PXR_SEQ}\n- ligand:\n    id: B\n    smiles: CCOc1ccccc1\nproperties:\n- affinity:\n    binder: B\n'
Path('/kaggle/working/test.yaml').write_text(yaml_str)
T(f'Wrote test.yaml ({len(yaml_str)} chars)')
OUT = Path('/kaggle/working/o_test')
OUT.mkdir(exist_ok=True)
cmd = [BIN, 'predict', '/kaggle/working/test.yaml', '--out_dir', str(OUT),
       '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
T(f'CMD: {" ".join(cmd)}')
T('Running boltz (max 30 min)...')
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=1800)
T(f'rc={r.returncode} elapsed={(time.time()-t0)/60:.1f}min')
T(f'  stdout[-2000:]: {r.stdout[-2000:]}')
if r.returncode != 0:
    T(f'  stderr[-2000:]: {r.stderr[-2000:]}')
import json
for jf in OUT.rglob('*affinity*.json'):
    T(f'  AFF FILE: {jf}: {open(jf).read()[:500]}')
T(f'All output files: {[str(p.relative_to(OUT)) for p in OUT.rglob("*")][:30]}')
T('=== CELL_4_END ===')
TRACE.close()